# Phase 10 — RAG Knowledge Base, Vector Indexing & Grounded Retrieval

**Cyber Threat Decision Support Pipeline — Phase 10**

**Status:** ✅ Stage B Executed & Audited  
**Corpus Sources:** MITRE ATT&CK Enterprise v15.1, NIST SP 800-61r2, NVD CVE-2014-0160  
**Embedding Model:** `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional, CPU-only)  
**Vector Database:** ChromaDB v1.5.9 (Rust CFFI `abi3` engine, persistent local storage)  
**Evaluation Partitions:** Dev: 40 queries, Test: 54 queries, Blind: 15 queries

---

## Executive Summary & Research Methodology
Phase 10 provides the grounded cybersecurity domain knowledge layer to support security analysts in investigating alerts detected by Phase 5 XGBoost, flagged by Phase 7 rules, and grouped by Phase 8 event correlation.

### Core Scientific Principles & Controls:
1. **Primary Sources Only:** Exclusively trusted industry authorities (MITRE ATT&CK v15.1, NIST SP 800-61r2, NVD). Zero LLM-generated corpus text.
2. **Tokenizer-Enforced Chunking:** Hard 256-token limit based on the embedding model's own WordPiece tokenizer to eliminate silent embedding truncation.
3. **Clean Oracle Audit:** Formal separation of predicted labels from true labels. Unfiltered semantic search evaluated strictly on observable queries; mapped retrieval evaluated strictly on predicted labels.
4. **Zero-Embedding Serving Path:** Fast, lightweight production serving path with ~58 MB peak RSS via deterministic label-to-technique mapping and cached chunks.


In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
rag_dir = root / 'artifacts' / 'rag'

# Load corpus manifest and chunk records
with open(rag_dir / 'corpus_manifest_v2.json', 'r', encoding='utf-8') as f:
    manifest = json.load(f)

with open(rag_dir / 'clean_reconciled_evaluation_v1.json', 'r', encoding='utf-8') as f:
    clean_eval = json.load(f)

with open(rag_dir / 'unified_serving_memory_profile_v1.json', 'r', encoding='utf-8') as f:
    unified_mem = json.load(f)

with open(rag_dir / 'fresh_process_memory_profiles_v1.json', 'r', encoding='utf-8') as f:
    fresh_mem = json.load(f)

print("=== PHASE 10 CORPUS AUDIT ===")
for src in manifest.get('sources', []):
    print(f"- {src.get('source_id', 'N/A')}: {src.get('name', 'N/A')} ({src.get('version', 'N/A')}) — {src.get('license_name', 'N/A')}")



## 1. Token-Capped Chunking & Corpus Geometry
Analysis of chunk size distributions, token count compliance, and technique coverage.


In [ ]:
with open(rag_dir / 'chunks_tok256_v1.json', 'r', encoding='utf-8') as f:
    chunks_tok256_data = json.load(f)

df_chunks = pd.DataFrame(chunks_tok256_data['chunks'])

print(f"Total Chunks Indexed (256-token cap): {len(df_chunks)}")
print(f"Unique ATT&CK Techniques Covered: {df_chunks['technique_id'].nunique()}")
print(f"Sources Represented: {df_chunks['source'].value_counts().to_dict()}")

fig, ax = plt.subplots(figsize=(8, 4))
df_chunks['token_count'].hist(bins=30, ax=ax, color='#34495e', edgecolor='white', alpha=0.8)
ax.axvline(256, color='#e74c3c', linestyle='--', linewidth=2, label='Tokenizer Cap (256)')
ax.set_title('Chunk Token Count Distribution (all-MiniLM-L6-v2 Tokenizer)', fontweight='bold')
ax.set_xlabel('Token Count')
ax.set_ylabel('Chunk Count')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 2. Reconciled Retrieval Evaluation (Clean Oracle Audit)
Comparing retrieval accuracy across Dev and Test splits with oracle contamination eliminated.


In [ ]:
dev_clean = clean_eval['part2_oracle_audit_summary']['dev_clean']
test_clean = clean_eval['part2_oracle_audit_summary']['test_clean']

eval_rows = [
    {
        'Split': 'Dev (n=26 threats)',
        'Pipeline Mode': 'Mapped on Stated Queries (n=12)',
        'Recall@1': dev_clean['mapped_on_stated_queries']['recall_at_1'],
        'Recall@5': dev_clean['mapped_on_stated_queries']['recall_at_5'],
        'MRR': dev_clean['mapped_on_stated_queries']['mrr']
    },
    {
        'Split': 'Dev (n=26 threats)',
        'Pipeline Mode': 'Semantic on Observable (n=14)',
        'Recall@1': dev_clean['semantic_on_observable_queries']['recall_at_1'],
        'Recall@5': dev_clean['semantic_on_observable_queries']['recall_at_5'],
        'MRR': dev_clean['semantic_on_observable_queries']['mrr']
    },
    {
        'Split': 'Dev (n=26 threats)',
        'Pipeline Mode': 'End-to-End Pipeline (All 26)',
        'Recall@1': dev_clean['end_to_end_pipeline_all_threats']['recall_at_1'],
        'Recall@5': dev_clean['end_to_end_pipeline_all_threats']['recall_at_5'],
        'MRR': dev_clean['end_to_end_pipeline_all_threats']['mrr']
    },
    {
        'Split': 'Dev (n=26 threats)',
        'Pipeline Mode': 'Realistic Label Lookup Baseline',
        'Recall@1': dev_clean['realistic_label_lookup_baseline']['recall_at_1'],
        'Recall@5': dev_clean['realistic_label_lookup_baseline']['recall_at_5'],
        'MRR': dev_clean['realistic_label_lookup_baseline']['mrr']
    },
    {
        'Split': 'Test (n=38 threats)',
        'Pipeline Mode': 'Mapped on Stated Queries (n=21)',
        'Recall@1': test_clean['mapped_on_stated_queries']['recall_at_1'],
        'Recall@5': test_clean['mapped_on_stated_queries']['recall_at_5'],
        'MRR': test_clean['mapped_on_stated_queries']['mrr']
    },
    {
        'Split': 'Test (n=38 threats)',
        'Pipeline Mode': 'Semantic on Observable (n=17)',
        'Recall@1': test_clean['semantic_on_observable_queries']['recall_at_1'],
        'Recall@5': test_clean['semantic_on_observable_queries']['recall_at_5'],
        'MRR': test_clean['semantic_on_observable_queries']['mrr']
    },
    {
        'Split': 'Test (n=38 threats)',
        'Pipeline Mode': 'End-to-End Pipeline (All 38)',
        'Recall@1': test_clean['end_to_end_pipeline_all_threats']['recall_at_1'],
        'Recall@5': test_clean['end_to_end_pipeline_all_threats']['recall_at_5'],
        'MRR': test_clean['end_to_end_pipeline_all_threats']['mrr']
    },
    {
        'Split': 'Test (n=38 threats)',
        'Pipeline Mode': 'Realistic Label Lookup Baseline',
        'Recall@1': test_clean['realistic_label_lookup_baseline']['recall_at_1'],
        'Recall@5': test_clean['realistic_label_lookup_baseline']['recall_at_5'],
        'MRR': test_clean['realistic_label_lookup_baseline']['mrr']
    }
]

df_eval = pd.DataFrame(eval_rows)
print("=== RECONCILED RETRIEVAL PERFORMANCE (CLEAN ORACLE AUDIT) ===")
print(df_eval.to_string(index=False))


## 3. MRR Discrepancy Reconciliation (Top-10 vs Top-5)
Audit of the dev MRR discrepancy between Stage B (0.3811) and Closure Pass 2 (0.3635).


In [ ]:
mrr_reconcil = clean_eval['part1_mrr_reconciliation']
print("=== MRR RECONCILIATION SUMMARY ===")
print(f"Stage B Dev MRR (top_k=10): {mrr_reconcil['dev_mrr_top10']:.4f}")
print(f"Closure Pass 2 Dev MRR (top_k=5): {mrr_reconcil['dev_mrr_top5']:.4f}")
print(f"Delta: {mrr_reconcil['delta']:.4f}")
print(f"Mathematical Cause: {mrr_reconcil['explanation']}")
print("\nDiscrepant Queries Awarded Reciprocal Rank in Top-10 but not Top-5:")
for q in mrr_reconcil['discrepant_queries']:
    print(f"- [{q['id']}] {q['text'][:60]}... -> First hit at Rank {q['rank_top10']} (RR: {q['rr_top10']:.4f})")


## 4. Serving Memory Profiles: Zero-Embedding vs. Full Semantic Search
Proving adherence to the < 500 MB RAM budget under both serving strategies.


In [ ]:
subsystems = []
peak_rss = []

for k, v in fresh_mem.items():
    subsystems.append(k.replace('_', ' ').title())
    peak_rss.append(v['peak_mb'])

subsystems.append('FastAPI Unified Serving')
peak_rss.append(unified_mem['peak_working_set_mb'])

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#27ae60' if p <= 500 else '#e74c3c' for p in peak_rss]
bars = ax.bar(subsystems, peak_rss, color=colors, width=0.5)
ax.axhline(500, color='red', linestyle='--', label='Memory Budget Ceiling (500 MB)')

ax.set_ylabel('Peak Working Set RSS (MB)', fontweight='bold')
ax.set_title('Subsystem Memory Profiles vs. 500 MB Budget Ceiling', fontweight='bold')
plt.xticks(rotation=25, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, y + 10, f"{y:.1f} MB", ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Unified Serving Peak RSS: {unified_mem['peak_working_set_mb']:.2f} MB")
print(f"Headroom below 500 MB ceiling: {500.0 - unified_mem['peak_working_set_mb']:.2f} MB")
